In [ ]:
# !pip install -q pydantic

In [ ]:
import os

# PREDICTIONS_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/predictions.jsonl"
PREDICTIONS_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/unified-sft-v1_test/predictions.jsonl"

assert os.path.exists(PREDICTIONS_PATH), f"File not found: {PREDICTIONS_PATH}"
print(f"Found predictions file: {PREDICTIONS_PATH}")
print(f"File size: {os.path.getsize(PREDICTIONS_PATH) / 1e6:.2f} MB")

Found predictions file: /content/drive/MyDrive/vlm-finetuning-project1/results/inference/unified-sft-v1_test/predictions.jsonl
File size: 3.47 MB


In [ ]:
import json

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[WARNING] Skipping unreadable line {line_num} in the .jsonl file itself: {e}")
    return records

records = load_jsonl(PREDICTIONS_PATH)
print(f"Loaded {len(records)} total records.")
print("\nSample record keys:", list(records[0].keys()) if records else "NO RECORDS")

Loaded 3004 total records.

Sample record keys: ['image_id', 'raw_output', 'sample', 'latency_seconds']


In [ ]:
import re
from typing import Any, Dict, Optional

def strip_fences(text: str) -> str:
    """Strips markdown code fences (```json ... ```) from a string.
    Uses regex to extract content between fences, ignoring any pre/post text."""
    match = re.search(r"```(?:json)?(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text.strip()


def parse_model_output(raw_str: str) -> Optional[Dict[str, Any]]:
    """Parses a raw string from the VLM into a dict. Returns None on failure."""
    if not raw_str or not raw_str.strip():
        return None
    text = strip_fences(raw_str)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


# Run Gate 1 over every record
parsed_results = []
for r in records:
    raw = r.get("raw_output", "")
    parsed = parse_model_output(raw)
    parsed_results.append(parsed)

n_json_valid = sum(1 for p in parsed_results if p is not None)
n_json_invalid = len(records) - n_json_valid

print("=" * 60)
print("GATE 1 — JSON PARSING")
print("=" * 60)
print(f"Total records:        {len(records)}")
print(f"Valid JSON:           {n_json_valid}  ({n_json_valid / len(records) * 100:.2f}%)")
print(f"Invalid JSON:         {n_json_invalid}  ({n_json_invalid / len(records) * 100:.2f}%)")

GATE 1 — JSON PARSING
Total records:        3004
Valid JSON:           2970  (98.87%)
Invalid JSON:         34  (1.13%)


In [ ]:
from pydantic import BaseModel, Field, conlist
from typing import List, Optional

BBox = conlist(float, min_length=4, max_length=4)


class RuleViolation(BaseModel):
    bounding_box: Optional[List[BBox]] = None
    reason: Optional[str] = None


class UnifiedOutput(BaseModel):
    """Mirrors data/schemas.py::UnifiedOutput — the model's required output contract."""
    caption: str
    rule_1_violation: Optional[RuleViolation] = None
    rule_2_violation: Optional[RuleViolation] = None
    rule_3_violation: Optional[RuleViolation] = None
    rule_4_violation: Optional[RuleViolation] = None
    excavator: List[BBox] = Field(default_factory=list)
    rebar: List[BBox] = Field(default_factory=list)
    worker_with_white_hard_hat: List[BBox] = Field(default_factory=list)


def validate_unified_output(parsed_data: Optional[Dict[str, Any]]):
    """Validates a parsed dict against the UnifiedOutput schema. Returns None on failure."""
    if parsed_data is None:
        return None
    try:
        return UnifiedOutput(**parsed_data)
    except Exception:
        return None


# Run Gate 2 over every record that passed Gate 1
schema_results = []
schema_errors = []  # keep the actual pydantic error message for diagnostics

for i, parsed in enumerate(parsed_results):
    if parsed is None:
        schema_results.append(None)
        schema_errors.append(None)
        continue
    try:
        validated = UnifiedOutput(**parsed)
        schema_results.append(validated)
        schema_errors.append(None)
    except Exception as e:
        schema_results.append(None)
        schema_errors.append(str(e))

n_schema_valid = sum(1 for s in schema_results if s is not None)
n_schema_invalid = len(records) - n_schema_valid

print("=" * 60)
print("GATE 2 — SCHEMA VALIDATION")
print("=" * 60)
print(f"Total records:            {len(records)}")
print(f"Valid schema:             {n_schema_valid}  ({n_schema_valid / len(records) * 100:.2f}%)")
print(f"Invalid schema:           {n_schema_invalid}  ({n_schema_invalid / len(records) * 100:.2f}%)")
print(f"  (of which failed Gate 1 already): {n_json_invalid}")
print(f"  (valid JSON but bad schema):      {n_schema_invalid - n_json_invalid}")

GATE 2 — SCHEMA VALIDATION
Total records:            3004
Valid schema:             2969  (98.83%)
Invalid schema:           35  (1.17%)
  (of which failed Gate 1 already): 34
  (valid JSON but bad schema):      1


In [ ]:
print("=" * 60)
print("COMBINED FUNNEL SUMMARY")
print("=" * 60)
print(f"{'Stage':<30} {'Count':>10} {'% of total':>12}")
print("-" * 54)
print(f"{'Total records':<30} {len(records):>10} {'100.00%':>12}")
print(f"{'Passed Gate 1 (JSON)':<30} {n_json_valid:>10} {n_json_valid/len(records)*100:>11.2f}%")
print(f"{'Passed Gate 2 (Schema)':<30} {n_schema_valid:>10} {n_schema_valid/len(records)*100:>11.2f}%")
print(f"{'Failed Gate 1 (JSON)':<30} {n_json_invalid:>10} {n_json_invalid/len(records)*100:>11.2f}%")
print(f"{'Failed Gate 2 only':<30} {n_schema_invalid - n_json_invalid:>10} {(n_schema_invalid - n_json_invalid)/len(records)*100:>11.2f}%")

if n_json_invalid == 0 and n_schema_invalid == 0:
    print("\n✅ ALL records passed both gates cleanly.")
else:
    print(f"\n⚠️  {n_schema_invalid} / {len(records)} records will NOT contribute to strict-vs-valid")
    print("    metric differences downstream — inspect the failure breakdowns below.")

COMBINED FUNNEL SUMMARY
Stage                               Count   % of total
------------------------------------------------------
Total records                        3004      100.00%
Passed Gate 1 (JSON)                 2970       98.87%
Passed Gate 2 (Schema)               2969       98.83%
Failed Gate 1 (JSON)                   34        1.13%
Failed Gate 2 only                      1        0.03%

⚠️  35 / 3004 records will NOT contribute to strict-vs-valid
    metric differences downstream — inspect the failure breakdowns below.


In [ ]:
from collections import Counter

# Only look at records that had valid JSON but still failed schema
schema_only_failures = [
    (i, schema_errors[i]) for i in range(len(records))
    if parsed_results[i] is not None and schema_results[i] is None
]

print(f"Records with valid JSON but invalid schema: {len(schema_only_failures)}\n")

if schema_only_failures:
    # Extract the pydantic "field required" / "type" style short reason for grouping
    def short_reason(err_msg: str) -> str:
        lines = err_msg.strip().split("\n")
        # pydantic v2 errors look like: "<field>\n  <message> [type=..., ...]"
        reasons = [l.strip() for l in lines if "[type=" in l]
        return reasons[0].split("[type=")[0].strip() if reasons else lines[-1][:80]

    reason_counts = Counter(short_reason(err) for _, err in schema_only_failures)

    print("Most common schema-failure reasons:")
    for reason, count in reason_counts.most_common(10):
        print(f"  {count:>4}x  {reason}")
else:
    print("No schema-only failures — every valid-JSON record also matched the schema.")

Records with valid JSON but invalid schema: 1

Most common schema-failure reasons:
     1x  List should have at least 4 items after validation, not 2


In [ ]:
N_EXAMPLES = 50

print("=" * 60)
print(f"GATE 1 FAILURE EXAMPLES (up to {N_EXAMPLES})")
print("=" * 60)
gate1_failures = [i for i in range(len(records)) if parsed_results[i] is None]
for idx in gate1_failures[:N_EXAMPLES]:
    r = records[idx]
    print(f"\n--- image_id: {r.get('image_id', 'UNKNOWN')} (record #{idx}) ---")
    raw = r.get("raw_output", "")
    print(raw)

print("\n" + "=" * 60)
print(f"GATE 2 FAILURE EXAMPLES (up to {N_EXAMPLES})")
print("=" * 60)
gate2_only_failures = [i for i, _ in schema_only_failures]
for idx in gate2_only_failures[:N_EXAMPLES]:
    r = records[idx]
    print(f"\n--- image_id: {r.get('image_id', 'UNKNOWN')} (record #{idx}) ---")
    print("Parsed JSON:", json.dumps(parsed_results[idx], indent=2))
    print("Pydantic error:", schema_errors[idx])

GATE 1 FAILURE EXAMPLES (up to 50)

--- image_id: 0002547 (record #48) ---
```json
{"caption":"The image shows multiple rebars laid out on the ground with some standing upright and others lying horizontally. There is one worker walking across the rebar area towards the right side.","rule_1_violation":null,"rule_2_violation":null,"rule_3_violation":null,"rule_4_violation":null,"excavator":[],"rebar":[[680,590,740,640],[100,240,270,330],[160,150,270,200],[100,20,190,70],[10,650,320,850],[300,370,580,440],[300,200,400,310],[400,180,480,260],[480,140,550,260],[550,120,600,220],[890,170,990,280],[940,270,1000,380],[10,30,100,70],[10,10,100,40],[10,40,100,80],[10,80,100,120],[10,120,100,160],[10,160,100,200],[10,200,100,240],[10,240,100,280],[10,30,100,70],[10,70,100,110],[10,110,100,150],[10,150,100,190],[10,190,100,230],[10,230,100,270],[10,270,100,310],[10,310,100,350],[10,350,100,390],[10,400,100,440],[10,440,100,480],[10,480,100,520],[10,520,100,560],[10,560,100,600],[10,600,100,640],[1

In [ ]:
pass_rate = n_schema_valid / len(records) * 100

print("=" * 60)
print("VALIDATION VERDICT")
print("=" * 60)
print(f"Total records checked:     {len(records)}")
print(f"Fully valid (both gates):  {n_schema_valid}  ({pass_rate:.2f}%)")
print(f"Failed at least one gate: {len(records) - n_schema_valid}  ({100 - pass_rate:.2f}%)")

if pass_rate >= 95:
    print("\n✅ High pass rate — predictions file looks healthy for downstream evaluation.")
elif pass_rate >= 80:
    print("\n⚠️  Moderate failure rate — worth inspecting failure examples above before")
    print("    trusting strict-pass metrics; valid-pass metrics will still be reliable.")
else:
    print("\n❌ High failure rate — something may be wrong with generation (e.g. truncation,")
    print("    prompt drift). Inspect failure examples above.")

VALIDATION VERDICT
Total records checked:     3004
Fully valid (both gates):  2969  (98.83%)
Failed at least one gate: 35  (1.17%)

✅ High pass rate — predictions file looks healthy for downstream evaluation.


In [1]:
# !pip install -q pydantic

In [2]:
import os

# PREDICTIONS_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/predictions.jsonl"
PREDICTIONS_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/repair_applied/predictions_repaired.jsonl"
# PREDICTIONS_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/unified-sft-v1_test/predictions.jsonl"

assert os.path.exists(PREDICTIONS_PATH), f"File not found: {PREDICTIONS_PATH}"
print(f"Found predictions file: {PREDICTIONS_PATH}")
print(f"File size: {os.path.getsize(PREDICTIONS_PATH) / 1e6:.2f} MB")

Found predictions file: /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/repair_applied/predictions_repaired.jsonl
File size: 7.94 MB


In [3]:
import json

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[WARNING] Skipping unreadable line {line_num} in the .jsonl file itself: {e}")
    return records

records = load_jsonl(PREDICTIONS_PATH)
print(f"Loaded {len(records)} total records.")
print("\nSample record keys:", list(records[0].keys()) if records else "NO RECORDS")

Loaded 3004 total records.

Sample record keys: ['image_id', 'raw_output', 'sample', 'latency_seconds', 'original_raw_output', 'repair_status']


In [4]:
import re
from typing import Any, Dict, Optional

def strip_fences(text: str) -> str:
    """Strips markdown code fences (```json ... ```) from a string.
    Uses regex to extract content between fences, ignoring any pre/post text."""
    match = re.search(r"```(?:json)?(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text.strip()


def parse_model_output(raw_str: str) -> Optional[Dict[str, Any]]:
    """Parses a raw string from the VLM into a dict. Returns None on failure."""
    if not raw_str or not raw_str.strip():
        return None
    text = strip_fences(raw_str)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


# Run Gate 1 over every record
parsed_results = []
for r in records:
    raw = r.get("raw_output", "")
    parsed = parse_model_output(raw)
    parsed_results.append(parsed)

n_json_valid = sum(1 for p in parsed_results if p is not None)
n_json_invalid = len(records) - n_json_valid

print("=" * 60)
print("GATE 1 — JSON PARSING")
print("=" * 60)
print(f"Total records:        {len(records)}")
print(f"Valid JSON:           {n_json_valid}  ({n_json_valid / len(records) * 100:.2f}%)")
print(f"Invalid JSON:         {n_json_invalid}  ({n_json_invalid / len(records) * 100:.2f}%)")

GATE 1 — JSON PARSING
Total records:        3004
Valid JSON:           2993  (99.63%)
Invalid JSON:         11  (0.37%)


In [5]:
from pydantic import BaseModel, Field, conlist
from typing import List, Optional

BBox = conlist(float, min_length=4, max_length=4)


class RuleViolation(BaseModel):
    bounding_box: Optional[List[BBox]] = None
    reason: Optional[str] = None


class UnifiedOutput(BaseModel):
    """Mirrors data/schemas.py::UnifiedOutput — the model's required output contract."""
    caption: str
    rule_1_violation: Optional[RuleViolation] = None
    rule_2_violation: Optional[RuleViolation] = None
    rule_3_violation: Optional[RuleViolation] = None
    rule_4_violation: Optional[RuleViolation] = None
    excavator: List[BBox] = Field(default_factory=list)
    rebar: List[BBox] = Field(default_factory=list)
    worker_with_white_hard_hat: List[BBox] = Field(default_factory=list)


def validate_unified_output(parsed_data: Optional[Dict[str, Any]]):
    """Validates a parsed dict against the UnifiedOutput schema. Returns None on failure."""
    if parsed_data is None:
        return None
    try:
        return UnifiedOutput(**parsed_data)
    except Exception:
        return None


# Run Gate 2 over every record that passed Gate 1
schema_results = []
schema_errors = []  # keep the actual pydantic error message for diagnostics

for i, parsed in enumerate(parsed_results):
    if parsed is None:
        schema_results.append(None)
        schema_errors.append(None)
        continue
    try:
        validated = UnifiedOutput(**parsed)
        schema_results.append(validated)
        schema_errors.append(None)
    except Exception as e:
        schema_results.append(None)
        schema_errors.append(str(e))

n_schema_valid = sum(1 for s in schema_results if s is not None)
n_schema_invalid = len(records) - n_schema_valid

print("=" * 60)
print("GATE 2 — SCHEMA VALIDATION")
print("=" * 60)
print(f"Total records:            {len(records)}")
print(f"Valid schema:             {n_schema_valid}  ({n_schema_valid / len(records) * 100:.2f}%)")
print(f"Invalid schema:           {n_schema_invalid}  ({n_schema_invalid / len(records) * 100:.2f}%)")
print(f"  (of which failed Gate 1 already): {n_json_invalid}")
print(f"  (valid JSON but bad schema):      {n_schema_invalid - n_json_invalid}")

GATE 2 — SCHEMA VALIDATION
Total records:            3004
Valid schema:             2993  (99.63%)
Invalid schema:           11  (0.37%)
  (of which failed Gate 1 already): 11
  (valid JSON but bad schema):      0


In [6]:
print("=" * 60)
print("COMBINED FUNNEL SUMMARY")
print("=" * 60)
print(f"{'Stage':<30} {'Count':>10} {'% of total':>12}")
print("-" * 54)
print(f"{'Total records':<30} {len(records):>10} {'100.00%':>12}")
print(f"{'Passed Gate 1 (JSON)':<30} {n_json_valid:>10} {n_json_valid/len(records)*100:>11.2f}%")
print(f"{'Passed Gate 2 (Schema)':<30} {n_schema_valid:>10} {n_schema_valid/len(records)*100:>11.2f}%")
print(f"{'Failed Gate 1 (JSON)':<30} {n_json_invalid:>10} {n_json_invalid/len(records)*100:>11.2f}%")
print(f"{'Failed Gate 2 only':<30} {n_schema_invalid - n_json_invalid:>10} {(n_schema_invalid - n_json_invalid)/len(records)*100:>11.2f}%")

if n_json_invalid == 0 and n_schema_invalid == 0:
    print("\n✅ ALL records passed both gates cleanly.")
else:
    print(f"\n⚠️  {n_schema_invalid} / {len(records)} records will NOT contribute to strict-vs-valid")
    print("    metric differences downstream — inspect the failure breakdowns below.")

COMBINED FUNNEL SUMMARY
Stage                               Count   % of total
------------------------------------------------------
Total records                        3004      100.00%
Passed Gate 1 (JSON)                 2993       99.63%
Passed Gate 2 (Schema)               2993       99.63%
Failed Gate 1 (JSON)                   11        0.37%
Failed Gate 2 only                      0        0.00%

⚠️  11 / 3004 records will NOT contribute to strict-vs-valid
    metric differences downstream — inspect the failure breakdowns below.


In [7]:
from collections import Counter

# Only look at records that had valid JSON but still failed schema
schema_only_failures = [
    (i, schema_errors[i]) for i in range(len(records))
    if parsed_results[i] is not None and schema_results[i] is None
]

print(f"Records with valid JSON but invalid schema: {len(schema_only_failures)}\n")

if schema_only_failures:
    # Extract the pydantic "field required" / "type" style short reason for grouping
    def short_reason(err_msg: str) -> str:
        lines = err_msg.strip().split("\n")
        # pydantic v2 errors look like: "<field>\n  <message> [type=..., ...]"
        reasons = [l.strip() for l in lines if "[type=" in l]
        return reasons[0].split("[type=")[0].strip() if reasons else lines[-1][:80]

    reason_counts = Counter(short_reason(err) for _, err in schema_only_failures)

    print("Most common schema-failure reasons:")
    for reason, count in reason_counts.most_common(10):
        print(f"  {count:>4}x  {reason}")
else:
    print("No schema-only failures — every valid-JSON record also matched the schema.")

Records with valid JSON but invalid schema: 0

No schema-only failures — every valid-JSON record also matched the schema.


In [8]:
N_EXAMPLES = 50

print("=" * 60)
print(f"GATE 1 FAILURE EXAMPLES (up to {N_EXAMPLES})")
print("=" * 60)
gate1_failures = [i for i in range(len(records)) if parsed_results[i] is None]
for idx in gate1_failures[:N_EXAMPLES]:
    r = records[idx]
    print(f"\n--- image_id: {r.get('image_id', 'UNKNOWN')} (record #{idx}) ---")
    raw = r.get("raw_output", "")
    print(raw)

print("\n" + "=" * 60)
print(f"GATE 2 FAILURE EXAMPLES (up to {N_EXAMPLES})")
print("=" * 60)
gate2_only_failures = [i for i, _ in schema_only_failures]
for idx in gate2_only_failures[:N_EXAMPLES]:
    r = records[idx]
    print(f"\n--- image_id: {r.get('image_id', 'UNKNOWN')} (record #{idx}) ---")
    print("Parsed JSON:", json.dumps(parsed_results[idx], indent=2))
    print("Pydantic error:", schema_errors[idx])

GATE 1 FAILURE EXAMPLES (up to 50)

--- image_id: 0003948 (record #61) ---
```json
{
  "caption": "A nighttime scene showing a concrete mixer truck with its long boom extended towards a roadwork site. The area is illuminated by streetlights, and there's a temporary barrier on one side. A worker wearing a white hard hat stands near the right edge of the frame.",
  "rule_1_violation": {
    "bounding_box": [
      "758,690,800,700,
      800,700,800,700,
      800,700,800,700,
      800,700,800,700
    ],
    "reason": "The worker is not wearing any protective gear other than a white hard hat."
  },
  "rule_2_violation": null,
  "rule_3_violation": null,
  "rule_4_violation": null,
  "excavator": [],
  "rebar": [],
  "worker_with_white_hard_hat": [
    "988,690,1000,700
  ]
}
```

--- image_id: 0003478 (record #200) ---
```json
{
  "caption": "The image shows a construction site with a large pile of earth in the background, green trees in the middle ground, and scaffolding on the right s

In [9]:
pass_rate = n_schema_valid / len(records) * 100

print("=" * 60)
print("VALIDATION VERDICT")
print("=" * 60)
print(f"Total records checked:     {len(records)}")
print(f"Fully valid (both gates):  {n_schema_valid}  ({pass_rate:.2f}%)")
print(f"Failed at least one gate: {len(records) - n_schema_valid}  ({100 - pass_rate:.2f}%)")

if pass_rate >= 95:
    print("\n✅ High pass rate — predictions file looks healthy for downstream evaluation.")
elif pass_rate >= 80:
    print("\n⚠️  Moderate failure rate — worth inspecting failure examples above before")
    print("    trusting strict-pass metrics; valid-pass metrics will still be reliable.")
else:
    print("\n❌ High failure rate — something may be wrong with generation (e.g. truncation,")
    print("    prompt drift). Inspect failure examples above.")

VALIDATION VERDICT
Total records checked:     3004
Fully valid (both gates):  2993  (99.63%)
Failed at least one gate: 11  (0.37%)

✅ High pass rate — predictions file looks healthy for downstream evaluation.
